# Exploratory Data Analysis

This notebook loads the raw StreamWave subscriber and watch-history datasets for initial analysis.

In [ ]:
from pathlib import Path
import pandas as pd

root_dir = Path.cwd()
raw_data_dir = root_dir / "data" / "raw"

subscribers_path = raw_data_dir / "subscribers.csv"
watch_history_path = raw_data_dir / "watch_history.csv"

raw_subscribers = pd.read_csv(subscribers_path)
raw_watch_history = pd.read_csv(watch_history_path)


def audit_dataframe(df, name):
    lines = [f"=== {name} ==="]
    lines.append(f"Rows: {df.shape[0]}")
    lines.append(f"Columns: {df.shape[1]}")
    lines.append("Data types:")
    lines.append(df.dtypes.astype(str).to_string())
    lines.append("Missing values:")
    lines.append(df.isna().sum().to_string())
    lines.append("Duplicate rows:")
    lines.append(str(df.duplicated().sum()))
    lines.append("")
    return "\n".join(lines)

report = audit_dataframe(raw_subscribers, "raw_subscribers")
report += audit_dataframe(raw_watch_history, "raw_watch_history")

print(report)

output_path = root_dir / "outputs" / "raw_data_quality_report.txt"
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(report)
print(f"\nReport saved to {output_path}")

# Prepare subscriber-level data for EDA
watch_engagement = (
    raw_watch_history[["subscriber_id", "engagement_score"]]
    .drop_duplicates(subset=["subscriber_id"])
    .copy()
)

merged_subscribers = raw_subscribers.merge(
    watch_engagement,
    on="subscriber_id",
    how="left",
    validate="one_to_one",
)

# Parse date columns to datetime
date_columns = [col for col in merged_subscribers.columns if "date" in col.lower()]
for col in date_columns:
    merged_subscribers[col] = pd.to_datetime(merged_subscribers[col], errors="coerce")

# Do not impute churn_date; null indicates an active subscriber.
# Impute missing engagement_score values with 0 for users who have no watch sessions.
merged_subscribers["engagement_score"] = merged_subscribers["engagement_score"].fillna(0)

missing_engagement = merged_subscribers["engagement_score"].isna().sum()
print(f"Missing engagement_score values after imputation: {missing_engagement}")
root_dir = Path.cwd()
raw_data_dir = root_dir / "data" / "raw"

subscribers_path = raw_data_dir / "subscribers.csv"
watch_history_path = raw_data_dir / "watch_history.csv"

raw_subscribers = pd.read_csv(subscribers_path)
raw_watch_history = pd.read_csv(watch_history_path)

ENGAGEMENT_IMPUTATION_ASSERT_MSG = "There should be zero null values in engagement_score after imputation"


def audit_dataframe(df, name):
    lines = [f"=== {name} ==="]
    lines.append(f"Rows: {df.shape[0]}")
    lines.append(f"Columns: {df.shape[1]}")
    lines.append("Data types:")
    lines.append(df.dtypes.astype(str).to_string())
    lines.append("Missing values:")
    lines.append(df.isna().sum().to_string())
    lines.append("Duplicate rows:")
    lines.append(str(df.duplicated().sum()))
    lines.append("")
    return "\n".join(lines)


report = audit_dataframe(raw_subscribers, "raw_subscribers")
report += audit_dataframe(raw_watch_history, "raw_watch_history")

print(report)

output_path = root_dir / "outputs" / "raw_data_quality_report.txt"
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(report)
print(f"\nReport saved to {output_path}")

# Prepare subscriber-level data for EDA
watch_engagement = (
    raw_watch_history[["subscriber_id", "engagement_score"]]
    .drop_duplicates(subset=["subscriber_id"])
    .copy()
)

merged_subscribers = raw_subscribers.merge(
    watch_engagement,
    on="subscriber_id",
    how="left",
    validate="one_to_one",
)

# Parse date columns to datetime
date_columns = [col for col in merged_subscribers.columns if "date" in col.lower()]
for col in date_columns:
    merged_subscribers[col] = pd.to_datetime(merged_subscribers[col], errors="coerce")

# Do not impute churn_date; null indicates an active subscriber.
# Impute missing engagement_score values with 0 for users who have no watch sessions.
merged_subscribers["engagement_score"] = merged_subscribers["engagement_score"].fillna(0)

missing_engagement = merged_subscribers["engagement_score"].isna().sum()
print(f"Missing engagement_score values after imputation: {missing_engagement}")
assert missing_engagement == 0, ENGAGEMENT_IMPUTATION_ASSERT_MSG

print("\nMerged subscriber-level dataframe shape:", merged_subscribers.shape)
print("Parsed date columns:", date_columns)





merged_subscribers["engagement_score"] = merged_subscribers["engagement_score"].fillna(0)

missing_engagement = merged_subscribers["engagement_score"].isna().sum()
print(f"Missing engagement_score values after imputation: {missing_engagement}")
assert missing_engagement == 0, "There should be zero null values in engagement_score after imputation"

print("\nMerged subscriber-level dataframe shape:", merged_subscribers.shape)
print("Parsed date columns:", date_columns)

missing_engagement = merged_subscribers["engagement_score"].isna().sum()
print(f"Missing engagement_score values after imputation: {missing_engagement}")
assert missing_engagement == 0, "There should be zero null values in engagement_score after imputation"

print("\nMerged subscriber-level dataframe shape:", merged_subscribers.shape)
print("Parsed date columns:", date_columns)

In [ ]:
import sys
import matplotlib
if 'ipykernel' not in sys.modules:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

figures_dir = root_dir / "outputs" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

def save_fig(plot_path):
    plt.savefig(plot_path, dpi=300)
    if 'ipykernel' in sys.modules:
        plt.show()
    plt.close()

channel_counts = (
    merged_subscribers.groupby("signup_channel")["subscriber_id"]
    .nunique()
    .reset_index(name="subscriber_count")
    .sort_values("subscriber_count", ascending=False)
)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=channel_counts,
    x="signup_channel",
    y="subscriber_count",
    palette="viridis",
)
plt.title("Total Subscriber Count by Acquisition Channel")
plt.xlabel("Acquisition Channel")
plt.ylabel("Subscriber Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plot_path = figures_dir / "subscriber_count_by_acquisition_channel.png"
save_fig(plot_path)
print(f"Figure saved to {plot_path}")

In [ ]:
# Additional EDA plots: subscription tier, engagement distribution, and subscriber tenure
merged_subscribers = merged_subscribers.copy()

# Compute tenure in months from signup_date to churn_date or today
now = pd.Timestamp.now().normalize()
merged_subscribers["tenure_days"] = (
    merged_subscribers["churn_date"].fillna(now) - merged_subscribers["signup_date"]
).dt.days
merged_subscribers["tenure_months"] = merged_subscribers["tenure_days"] / 30.0

sns.set_style("whitegrid")

# Plot 1: Subscription tier distribution
plt.figure(figsize=(10, 6))
plan_counts = merged_subscribers["current_plan"].value_counts().reset_index()
plan_counts.columns = ["current_plan", "subscriber_count"]
sns.barplot(
    data=plan_counts,
    x="current_plan",
    y="subscriber_count",
    palette="coolwarm",
)
plt.title("Subscription Tier Distribution")
plt.xlabel("Subscription Tier")
plt.ylabel("Subscriber Count")
plt.tight_layout()
plot_path = figures_dir / "subscription_tier_distribution.png"
save_fig(plot_path)
print(f"Figure saved to {plot_path}")

# Plot 2: Engagement level distribution
plt.figure(figsize=(10, 6))
sns.histplot(
    merged_subscribers,
    x="engagement_score",
    bins=30,
    kde=True,
    color="#4c72b0",
)
plt.title("Engagement Score Distribution")
plt.xlabel("Engagement Score")
plt.ylabel("Subscriber Count")
plt.tight_layout()
plot_path = figures_dir / "engagement_level_distribution.png"
plt.savefig(plot_path, dpi=300)
plt.show()
save_fig(plot_path)
print(f"Figure saved to {plot_path}")

# Plot 3: Subscriber tenure distribution
plt.figure(figsize=(10, 6))
sns.histplot(
    merged_subscribers,
    x="tenure_months",
    bins=30,
    kde=True,
    color="#55a868",
)
plt.title("Subscriber Tenure Distribution")
plt.xlabel("Tenure (months)")
plt.ylabel("Subscriber Count")
plt.tight_layout()
plot_path = figures_dir / "subscriber_tenure_distribution.png"
plt.savefig(plot_path, dpi=300)
plt.show()
save_fig(plot_path)
print(f"Figure saved to {plot_path}")

In [ ]:
# Boxplots for engagement by channel, plan, and churn status
engagement_data = merged_subscribers.copy()
engagement_data["churn_status_label"] = engagement_data["churn_status"].map({0: "Retained", 1: "Churned"})

plt.figure(figsize=(12, 7))
sns.boxplot(
    data=engagement_data,
    x="signup_channel",
    y="engagement_score",
    palette="Set2",
)
plt.title("Engagement Level Distribution by Acquisition Channel")
plt.xlabel("Acquisition Channel")
plt.ylabel("Engagement Score")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plot_path = figures_dir / "engagement_by_acquisition_channel.png"
save_fig(plot_path)
print(f"Figure saved to {plot_path}")

plt.figure(figsize=(12, 7))
sns.boxplot(
    data=engagement_data,
    x="current_plan",
    y="engagement_score",
    palette="Set3",
)
plt.title("Engagement Level Distribution by Subscription Plan")
plt.xlabel("Subscription Plan")
plt.ylabel("Engagement Score")
plt.tight_layout()
plot_path = figures_dir / "engagement_by_subscription_plan.png"
plt.savefig(plot_path, dpi=300)
plt.show()
save_fig(plot_path)
print(f"Figure saved to {plot_path}")

plt.figure(figsize=(8, 7))
sns.boxplot(
    data=engagement_data,
    x="churn_status_label",
    y="engagement_score",
    palette=["#2ca02c", "#d62728"],
)
plt.title("Engagement Level Distribution by Churn Status")
plt.xlabel("Churn Status")
plt.ylabel("Engagement Score")
plt.tight_layout()
plot_path = figures_dir / "engagement_by_churn_status.png"
plt.savefig(plot_path, dpi=300)
plt.show()
save_fig(plot_path)
print(f"Figure saved to {plot_path}")

In [ ]:
# Engagement quantiles and churn-risk heatmaps
merged_subscribers = merged_subscribers.copy()
quantile_labels = [
    "Checked Out",
    "Slowing Down",
    "Casual",
    "Regular",
    "Obsessed",
]
merged_subscribers["engagement_tier"] = pd.qcut(
    merged_subscribers["engagement_score"],
    q=5,
    labels=quantile_labels,
    duplicates="drop",
)

heatmap_data_channel = (
    merged_subscribers
    .groupby(["engagement_tier", "signup_channel"])["churn_status"]
    .mean()
    .reset_index()
    .pivot(index="engagement_tier", columns="signup_channel", values="churn_status")
)

plt.figure(figsize=(12, 8))
sns.heatmap(
    heatmap_data_channel,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    cbar_kws={"label": "Churn Rate"},
)
plt.title("Churn Rate by Engagement Tier and Signup Channel")
plt.xlabel("Signup Channel")
plt.ylabel("Engagement Tier")
plt.tight_layout()
plot_path = figures_dir / "churn_heatmap_engagement_tier_signup_channel.png"
save_fig(plot_path)
print(f"Figure saved to {plot_path}")

heatmap_data_plan = (
    merged_subscribers
    .groupby(["engagement_tier", "current_plan"])["churn_status"]
    .mean()
    .reset_index()
    .pivot(index="engagement_tier", columns="current_plan", values="churn_status")
)

plt.figure(figsize=(12, 8))
sns.heatmap(
    heatmap_data_plan,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    cbar_kws={"label": "Churn Rate"},
)
plt.title("Churn Rate by Engagement Tier and Subscription Plan")
plt.xlabel("Subscription Plan")
plt.ylabel("Engagement Tier")
plt.tight_layout()
plot_path = figures_dir / "churn_heatmap_engagement_tier_subscription_plan.png"
plt.savefig(plot_path, dpi=300)
plt.show()
save_fig(plot_path)
print(f"Figure saved to {plot_path}")

In [ ]:
from scipy.stats import chi2_contingency

channel_data = merged_subscribers[merged_subscribers["signup_channel"] == "Free Trial"].copy()

ct = pd.crosstab(
    channel_data["engagement_tier"],
    channel_data["churn_status"],
    dropna=False,
)

print("Null hypothesis: Engagement tier and churn status are independent for Free Trial subscribers.")
print("Alternative hypothesis: Engagement tier and churn status are dependent for Free Trial subscribers.")
print()
print("Cross-tabulation (Engagement Tier x Churn Status):")
print(ct)
print()

chi2, p, dof, expected = chi2_contingency(ct)
print(f"Chi-square test statistic: {chi2:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"p-value: {p:.4f}")
print()
alpha = 0.05
if p < alpha:
    print(f"Conclusion: Reject the null hypothesis at alpha = {alpha}. Engagement tier and churn status are dependent for Free Trial subscribers.")
else:
    print(f"Conclusion: Fail to reject the null hypothesis at alpha = {alpha}. There is not enough evidence to conclude dependence between engagement tier and churn status for Free Trial subscribers.")

### Engagement Tier vs. Signup Channel Churn Risk

The heatmap shows whether high engagement tiers reduce churn risk across signup channels, and whether some channels still carry elevated churn risk even for more engaged subscribers.

### Engagement Tier vs. Subscription Plan Churn Risk

The second heatmap shows whether engagement tiers protect against churn risk in ad-supported or premium plans and which plan tiers remain vulnerable at lower engagement levels.

### Engagement by Subscription Plan

This box plot highlights differences in engagement across subscription plans, helping identify whether premium or lower-tier subscribers are more engaged.

### Engagement by Churn Status

Comparing retained versus churned users shows whether churned subscribers tend to have lower engagement scores than retained subscribers.